In [1]:
library(data.table)

# ============================================================
# BUILD LAGGED DATASET
#
# For each site-year Y, attach monthly climate and disturbance
# columns from Y-1 (lag1) and Y-2 (lag2).
#
# EFP anomaly lags are already in the dataset (from notebook 17).
# Traits are time-invariant and not lagged.
# ============================================================

input_file <- "derived_tables/outputs_afterEGU_results/EFP_mortality_trait_hydro_combined_with_EFPanom_memory.csv"
out_dir    <- "derived_tables/outputs_afterEGU_results"
out_file   <- file.path(out_dir, "EFP_mortality_trait_hydro_combined_with_meteo_dist_lags.csv")

In [2]:
# ------------------------------------------------------------
# 1) Load dataset
# ------------------------------------------------------------

dt <- fread(input_file)

cat("Input dataset:", nrow(dt), "rows x", ncol(dt), "cols\n")
cat("Sites:", uniqueN(dt$SITE_ID), "\n")
cat("Years:", min(dt$YEAR), "-", max(dt$YEAR), "\n")

Input dataset: 1180 rows x 237 cols
Sites: 184 
Years: 2017 - 2025 


In [3]:
# ------------------------------------------------------------
# 2) Identify columns to lag
# ------------------------------------------------------------

all_cols <- names(dt)

# monthly climate
meteo_cols <- grep("_(mean|p05|p95|sum)_M[0-9]{2}$", all_cols, value = TRUE)

# disturbance / mortality (all buffers, no _capped_)
dist_cols <- grep(
  paste0(
    "^(",
    "mortality_intensity_pct|",
    "deadwood_increase_sum_pp|",
    "deadwood_increase_area_frac|",
    "deadwood_increase_mean_pp|",
    "deadwood_mean_pct|",
    "loss_area_frac|",
    "loss_sum_pp|",
    "loss_mean_pp",
    ")_[0-9]+m$"
  ),
  all_cols, value = TRUE
)

cols_to_lag <- c(meteo_cols, dist_cols)

cat("Monthly climate cols to lag :", length(meteo_cols), "\n")
cat("Disturbance cols to lag     :", length(dist_cols), "\n")
cat("Total cols to lag           :", length(cols_to_lag), "\n")
cat("New columns added (x2 lags) :", length(cols_to_lag) * 2, "\n")

Monthly climate cols to lag : 132 
Disturbance cols to lag     : 40 
Total cols to lag           : 172 
New columns added (x2 lags) : 344 


In [4]:
# ------------------------------------------------------------
# 3) Build lag1 and lag2
#
# Shift YEAR by +lag so that when merged on SITE_ID+YEAR,
# row Y receives the values originally from row Y-lag.
# ------------------------------------------------------------

dt_out <- copy(dt)

for (lag in 1:2) {

  shifted <- dt[, c("SITE_ID", "YEAR", cols_to_lag), with = FALSE]
  shifted[, YEAR := YEAR + lag]

  lag_names <- paste0(cols_to_lag, "_lag", lag)
  setnames(shifted, cols_to_lag, lag_names)

  dt_out <- merge(dt_out, shifted, by = c("SITE_ID", "YEAR"), all.x = TRUE)

  cat("Lag", lag, "merged. Columns now:", ncol(dt_out), "\n")
}

cat("\nFinal dataset:", nrow(dt_out), "rows x", ncol(dt_out), "cols\n")

Lag 1 merged. Columns now: 409 
Lag 2 merged. Columns now: 581 

Final dataset: 1180 rows x 581 cols


In [5]:
# ------------------------------------------------------------
# 4) Coverage report
# ------------------------------------------------------------

lag1_probe <- paste0(meteo_cols[1], "_lag1")
lag2_probe <- paste0(meteo_cols[1], "_lag2")

n_lag1  <- sum(!is.na(dt_out[[lag1_probe]]))
n_lag2  <- sum(!is.na(dt_out[[lag2_probe]]))
n_total <- nrow(dt_out)

cat("Coverage:\n")
cat(sprintf("  lag1 available: %d / %d  (%.1f%%)\n", n_lag1, n_total, 100*n_lag1/n_total))
cat(sprintf("  lag2 available: %d / %d  (%.1f%%)\n", n_lag2, n_total, 100*n_lag2/n_total))

cat("\nRows missing lag1 by YEAR:\n")
print(dt_out[is.na(get(lag1_probe)), .N, by = YEAR][order(YEAR)])

cat("\nRows missing lag2 by YEAR:\n")
print(dt_out[is.na(get(lag2_probe)), .N, by = YEAR][order(YEAR)])

Coverage:
  lag1 available: 983 / 1180  (83.3%)
  lag2 available: 806 / 1180  (68.3%)

Rows missing lag1 by YEAR:
    YEAR     N
   <int> <int>
1:  2017   102
2:  2018    14
3:  2019    36
4:  2020    24
5:  2021    12
6:  2022     4
7:  2023     2
8:  2024     3

Rows missing lag2 by YEAR:
    YEAR     N
   <int> <int>
1:  2017   102
2:  2018   113
3:  2019    47
4:  2020    58
5:  2021    33
6:  2022    15
7:  2023     4
8:  2024     2


In [6]:
# ------------------------------------------------------------
# 5) Save + column manifest
# ------------------------------------------------------------

fwrite(dt_out, out_file)
cat("Saved:", out_file, "\n")
cat("Rows:", nrow(dt_out), "| Cols:", ncol(dt_out), "\n")

col_manifest <- data.table(
  column   = names(dt_out),
  category = fcase(
    names(dt_out) %in% paste0(meteo_cols, "_lag2"), "meteo_lag2",
    names(dt_out) %in% paste0(meteo_cols, "_lag1"), "meteo_lag1",
    names(dt_out) %in% meteo_cols,                   "meteo_current",
    names(dt_out) %in% paste0(dist_cols, "_lag2"),   "disturbance_lag2",
    names(dt_out) %in% paste0(dist_cols, "_lag1"),   "disturbance_lag1",
    names(dt_out) %in% dist_cols,                    "disturbance_current",
    grepl("_anom_lag", names(dt_out)),               "efp_anomaly_memory",
    default = "other"
  )
)

fwrite(col_manifest, file.path(out_dir, "lagged_dataset_column_manifest.csv"))
print(col_manifest[, .N, by = category][order(category)])

Saved: derived_tables/outputs_afterEGU_results/EFP_mortality_trait_hydro_combined_with_meteo_dist_lags.csv 
Rows: 1180 | Cols: 581 
              category     N
                <char> <int>
1: disturbance_current    40
2:    disturbance_lag1    40
3:    disturbance_lag2    40
4:  efp_anomaly_memory     8
5:       meteo_current   132
6:          meteo_lag1   132
7:          meteo_lag2   132
8:               other    57
